<!-- dd:dd-lesson-eo-1 -->

# Rearrange

*Einops · `eo-1`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# The side panel decides what you practise next. This cell only tells
# the notebook who you are, for the completion beacon.
DD_TOKEN = ""  # paste from the extension's Settings if you want beacons
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"
DD_LESSON_ID = "eo-1"


<!-- dd:dd-kp-einops-pattern-language -->

## The einops pattern language — naming and permuting axes

`einops.pattern-language`


`einops.rearrange(tensor, 'PATTERN')` is reshape/transpose with the axes
spelled out in words. The pattern is two axis lists around an arrow:

> `'b h w c -> b c h w'`
> — left side: name each input axis, in order. Right side: the same names,
> in the output's order.

Unlike einsum's one-letter subscripts, einops names are whole
space-separated WORDS (`batch`, `h`, `nh`), and — the big semantic
difference — **every name on the left must appear on the right** (rearrange
never sums; reducing is a different function, later KP). What rearrange
does is exactly what the name-shuffle says:

- `'b h w c -> b c h w'` — channels-last to channels-first: axis `c` moves
  to position 1, values untouched.
- `'b t d -> t b d'` — batch-first to time-first.
- `'h w -> w h'` — a 2-D transpose.

Why this beats `x.permute(0, 3, 1, 2)`: the pattern is
self-verifying documentation. It states what each axis MEANS, the library
checks that the input really has 4 axes, and six months later the intent is
still legible. In deep-learning code, layout bugs (bhwc vs bchw) are among
the most common and least visible — naming the axes at every hop is the
antidote, which is why the ARENA curriculum drills einops before touching
models.

Reading discipline (same ritual as einsum): identify each name's position
on the left (what it is) and on the right (where it goes). If a name
appears exactly once per side, the operation is a pure permutation — data
moves, nothing merges, splits, or disappears. Merging and splitting add
parentheses to this grammar — next two KPs.


Task: channels-last batch → channels-first, and batch-first sequence →
time-first — with one-element verification.


In [ ]:
import torch as t
import einops

arr = t.arange(24).reshape(2, 2, 2, 3)      # (b, h, w, c) channels-LAST

# Name the four axes; emit them with c pulled to the front block.
first = einops.rearrange(arr, 'b h w c -> b c h w')
assert first.shape == (2, 3, 2, 2)
# Track one element: input (b=1, h=0, w=1, c=2) must land at (1, 2, 0, 1).
assert arr[1, 0, 1, 2] == first[1, 2, 0, 1]

# Sequence layout swap: batch-first -> time-first.
seq = t.arange(12).reshape(2, 3, 2)         # (b, t, d)
tfirst = einops.rearrange(seq, 'b t d -> t b d')
assert tfirst.shape == (3, 2, 2)
assert seq[1, 2, 0] == tfirst[2, 1, 0]

# The pattern is checked against reality: wrong axis count = loud error.
try:
    einops.rearrange(seq, 'b h w c -> b c h w')   # 3-D data, 4-name pattern
    raised = False
except Exception as err:
    raised = True
    print("4-name pattern on 3-D data ->", type(err).__name__)
assert raised

print("channels-last", tuple(arr.shape), "-> channels-first",
      tuple(first.shape))
print("element (1,0,1,2) moved to (1,2,0,1):",
      arr[1, 0, 1, 2].item(), "==", first[1, 2, 0, 1].item())
print("batch-first", tuple(seq.shape), "-> time-first", tuple(tfirst.shape))




Why each step:

1. The tracked element (`arr[1,0,1,2] == first[1,2,0,1]`) is the same
   verification you used for einsum relayouts — indices permute exactly as
   the names did. One element proves the whole mapping.
2. Note what the names buy in the sequence example: 'b t d -> t b d' READS
   as "time first"; the transpose-tuple spelling `(1, 0, 2)` says the same
   thing to the machine and nothing to the reader.
3. The deliberate error shows einops as a shape CHECKER: patterns carry
   expectations, and mismatches fail at the call — not three functions
   later. This is a feature to lean on, not an annoyance.


In [ ]:
import torch as t
import einops

arr = t.arange(24).reshape(2, 2, 2, 3)      # (b, h, w, c) channels-LAST

# Name the four axes; emit them with c pulled to the front block.
first = einops.rearrange(arr, 'b h w c -> b c h w')
assert first.shape == (2, 3, 2, 2)
# Track one element: input (b=1, h=0, w=1, c=2) must land at (1, 2, 0, 1).
assert arr[1, 0, 1, 2] == first[1, 2, 0, 1]

# Sequence layout swap: batch-first -> time-first.
seq = t.arange(12).reshape(2, 3, 2)         # (b, t, d)
tfirst = einops.rearrange(seq, 'b t d -> t b d')
assert tfirst.shape == (3, 2, 2)
assert seq[1, 2, 0] == tfirst[2, 1, 0]

# The pattern is checked against reality: wrong axis count = loud error.
try:
    einops.rearrange(seq, 'b h w c -> b c h w')   # 3-D data, 4-name pattern
    raised = False
except Exception as err:
    raised = True
    print("4-name pattern on 3-D data ->", type(err).__name__)
assert raised

print("channels-last", tuple(arr.shape), "-> channels-first",
      tuple(first.shape))
print("element (1,0,1,2) moved to (1,2,0,1):",
      arr[1, 0, 1, 2].item(), "==", first[1, 2, 0, 1].item())
print("batch-first", tuple(seq.shape), "-> time-first", tuple(tfirst.shape))


<!-- dd:dd-q345 -->

### Problem 345 · faded

Channels-last batch to channels-first.


In [ ]:
import torch as t
import einops

def solve(arr):
    """(b, h, w, c) -> (b, c, h, w)."""
    return einops.rearrange(arr, '_____')


<!-- dd:dd-q388 -->

### Problem 388 · guided

Write a function solve(seq) that takes a batch-first sequence tensor (b, t, d) and returns the TIME-FIRST layout (t, b, d) — batch and time axes exchanged.


<details>
<summary>Hints</summary>

1. (b, t, d) to time-first (t, b, d) — name the three axes, reorder two of
   them.
2. All names appear on both sides — a pure permutation.
3. `'b t d -> t b d'`.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq):
    """Return the TIME-FIRST layout (t, b, d) — batch and time axes exchanged."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(2, 3, 1)))


<!-- dd:dd-q335 -->

### Problem 335 · independent

Write a function solve(img) that takes a channels-last image (h, w, c) and returns the channels-FIRST version (c, h, w).


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    """Return the channels-FIRST version (c, h, w)."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(1, 2, 3)))


<!-- dd:dd-q330 -->

### Problem 330 · independent

Write a function solve(arr) that takes a batch of shape (b, c, h, w) and returns shape (c, h, w, b) — the batch axis moved to the END, everything else in order.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (c, h, w, b) — the batch axis moved to the END, everything else in order."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(2, 1, 1, 2)))


<!-- dd:dd-q379 -->

### Problem 379 · independent

Write solve(imgs) for a (B, C, H, W) batch: swap the HEIGHT and WIDTH dimensions of every image (spatial transpose) — output (B, C, W, H). Pattern: 'b c h w -> b c w h'.


In [ ]:
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
imgs = arr

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


<!-- dd:dd-q319 -->

### Problem 319 · independent

Write a function solve(arr) that takes a channels-FIRST batch of shape (b, c, h, w) and returns the channels-LAST layout (b, h, w, c) — same values, the channel axis moved to the end.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return the channels-LAST layout (b, h, w, c) — same values, the channel axis moved to the end."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(1, 3, 1, 2)))


<!-- dd:dd-q327 -->

### Problem 327 · independent

Write a function solve(img) that takes a channels-last image of shape (h, w, c) and returns shape (h, c, w) — the channel and width axes exchanged, so color ends up between height and width.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    """Return shape (h, c, w) — the channel and width axes exchanged, so color ends up between height and width."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(1, 2, 3)))


<!-- dd:dd-q344 -->

### Problem 344 · independent

Write a function solve(img) that takes a channels-first image (c, h, w) and returns shape (c, w, h) — height and width TRANSPOSED within each channel, reflecting the image across its main diagonal.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img):
    """Return shape (c, w, h) — height and width TRANSPOSED within each channel, reflecting the image across its main"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(6.0).reshape(1, 2, 3)))


#### Common mistakes

- **"einops names are single characters like einsum."** — They're
  space-separated words: `'batch height width channels -> ...'` is legal and
  sometimes clearest. The space is the separator; 'bhwc' would be ONE axis
  named bhwc.
- **"rearrange can drop an axis I don't need."** — Every input name must
  appear in the output; rearrange is lossless by design. Dropping = summing
  or selecting, which are reduce (later KP) or plain indexing.
- **"It's just transpose with extra steps."** — It's transpose PLUS shape
  verification PLUS documentation. The pattern fails loudly when the input
  doesn't match the declared layout — the check you didn't know you needed
  until a bhwc/bchw bug eats an afternoon.


<!-- dd:dd-kp-einops-merge-axes -->

## Merging axes with (parentheses)

`einops.merge-axes`


Parentheses on the **output** side of a pattern MERGE axes into one:

> `'c h w -> c (h w)'`
> — h and w fuse into a single axis of length h·w.

Two things define what you get:

1. **Order inside the parens = nesting order.** The LEFT name varies
   slowest, the rightmost fastest (exactly row-major reshape). `(h w)`
   walks: h=0 with all w's, then h=1 with all w's… — each row in full, row
   by row. `(w h)` would walk columns instead. When a task says
   "row-major", "reading order", or "all of X's block before the next X",
   it is dictating the paren order.
2. **Which axes you merge — and they need not be adjacent in the input.**
   `'b h w c -> h (b w) c'` merges batch INTO width: because b is the slow
   (left) name, image 0's columns come first, then image 1's — the images
   laid side by side. Merging non-adjacent axes quietly includes the
   transpose that brings them together; the pattern spells the whole move.

The classic instances:

- Flatten spatial: `'c h w -> c (h w)'` — per-channel row-major flattening.
- Stack a batch vertically: `'b h w c -> (b h) w c'` — batch into height,
  image after image.
- Side-by-side concatenation: `'b h w c -> h (b w) c'`.
- Channel unroll: `'c h w -> (c h) w'` — all of channel 0's rows, then
  channel 1's… (c slow, h fast).

In raw PyTorch each of these is a permute+reshape pair you must derive;
in einops the pattern is the derivation.


Task: flatten an image's spatial axes per channel; lay a batch out side by
side.


In [ ]:
import torch as t
import einops

img = t.arange(12).reshape(3, 2, 2)      # (c, h, w)

# Merge h and w, h varying slowest: each channel flattens in reading order.
flat = einops.rearrange(img, 'c h w -> c (h w)')
assert flat.shape == (3, 4)
assert flat[0].tolist() == [0, 1, 2, 3]   # row 0 then row 1 of channel 0

# Paren order matters: (w h) reads DOWN the columns instead.
flat_cols = einops.rearrange(img, 'c h w -> c (w h)')
assert flat_cols[0].tolist() == [0, 2, 1, 3]

# Merge NON-adjacent axes: batch into width -> images side by side.
batch = t.arange(16).reshape(2, 2, 2, 2)  # (b, h, w, c)
wide = einops.rearrange(batch, 'b h w c -> h (b w) c')
assert wide.shape == (2, 4, 2)
# Row 0: image 0's two columns, THEN image 1's two columns (b is slow).
assert wide[0, :, 0].tolist() == [0, 2, 8, 10]
print("'(h w)' reads across rows:", flat[0])
print("'(w h)' reads down columns:", flat_cols[0])
print("batch merged into width", tuple(wide.shape), "-> row 0:",
      wide[0, :, 0])




Why each step:

1. The `(h w)` vs `(w h)` pair on the same data is the fastest way to burn
   in "left = slow": identical merge, different walk, different output.
   When unsure, test both on arange data — the values ARE their original
   positions.
2. For the side-by-side merge, predict before running: b slow means all of
   image 0's width before image 1's — that's "side by side, image 0 on the
   left". If you wanted interleaved columns, b would go FAST: `(w b)`.
3. Notice `wide`'s shape (2, 4, 2) contains the arithmetic (b·w = 4) — a
   merged axis's length is always the product, a free sanity check.


In [ ]:
import torch as t
import einops

img = t.arange(12).reshape(3, 2, 2)      # (c, h, w)

# Merge h and w, h varying slowest: each channel flattens in reading order.
flat = einops.rearrange(img, 'c h w -> c (h w)')
assert flat.shape == (3, 4)
assert flat[0].tolist() == [0, 1, 2, 3]   # row 0 then row 1 of channel 0

# Paren order matters: (w h) reads DOWN the columns instead.
flat_cols = einops.rearrange(img, 'c h w -> c (w h)')
assert flat_cols[0].tolist() == [0, 2, 1, 3]

# Merge NON-adjacent axes: batch into width -> images side by side.
batch = t.arange(16).reshape(2, 2, 2, 2)  # (b, h, w, c)
wide = einops.rearrange(batch, 'b h w c -> h (b w) c')
assert wide.shape == (2, 4, 2)
# Row 0: image 0's two columns, THEN image 1's two columns (b is slow).
assert wide[0, :, 0].tolist() == [0, 2, 8, 10]
print("'(h w)' reads across rows:", flat[0])
print("'(w h)' reads down columns:", flat_cols[0])
print("batch merged into width", tuple(wide.shape), "-> row 0:",
      wide[0, :, 0])


<!-- dd:dd-q391 -->

### Problem 391 · faded

Flatten spatial axes per channel, reading order.


In [ ]:
import torch as t
import einops

def solve(img):
    """(c, h, w) -> (c, h*w), rows before columns."""
    return einops.rearrange(img, '_____')


<!-- dd:dd-q347 -->

### Problem 347 · faded

Same input, a different merge: lay the channels out HORIZONTALLY, so channel
0's whole image sits left of channel 1's. Shape (c, h, w) -> (h, c·w). Two
decisions the flatten above did not ask for — WHICH pair of axes merges (they
are not adjacent in the input), and which of them is the slow one.


In [ ]:
import torch as t
import einops

def solve(img):
    """(c, h, w) -> (h, c*w): channel 0's image, then channel 1's, side by side."""
    return einops.rearrange(img, '_____')


<!-- dd:dd-q357 -->

### Problem 357 · guided

Write solve(img) for a (C, H, W) channels-first image: combine channels and height into one tall strip — output ((C·H), W), channel 0's rows first, then channel 1's, … Pattern: 'c h w -> (c h) w'.


<details>
<summary>Hints</summary>

1. (C, H, W) → ((C·H), W): channels and height merge into one tall strip,
   channel 0's rows first.
2. "Channel 0's rows first, then channel 1's" tells you which name is slow
   inside the parens.
3. `'c h w -> (c h) w'`.

</details>


In [ ]:
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
img = arr[0]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


<!-- dd:dd-q342 -->

### Problem 342 · independent

Write a function solve(arr) that takes a channels-last batch (b, h, w, c) and MERGES batch into height: return shape (b*h, w, c), all images stacked top-to-bottom in batch order — 'b h w c -> (b h) w c'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (b*h, w, c), all images stacked top-to-bottom in batch order — 'b h w c -> (b h) w c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q314 -->

### Problem 314 · independent

Write a function solve(arr) that takes a channels-last batch of shape (b, h, w, c) and concatenates the images SIDE BY SIDE: return shape (h, b*w, c) where image n occupies columns n*w through (n+1)*w - 1 — the einops pattern 'b h w c -> h (b w) c'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (h, b*w, c) where image n occupies columns n*w through (n+1)*w - 1 — the einops pattern 'b h w c """
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q346 -->

### Problem 346 · independent

Write a function solve(arr) that takes a channels-FIRST batch (b, c, h, w) and concatenates the images side by side WITHIN the channels-first layout: return shape (c, h, b*w) via 'b c h w -> c h (b w)'. (A companion drill does this for channels-last input.)


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (c, h, b*w) via 'b c h w -> c h (b w)'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q353 -->

### Problem 353 · independent

Write a function solve(seq_chunks) that takes a chunked sequence tensor of shape (b, n, p, d) — n chunks of p tokens each — and MERGES the chunk axes back into one sequence: return shape (b, n*p, d) via 'b n p d -> b (n p) d'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq_chunks):
    """Return shape (b, n*p, d) via 'b n p d -> b (n p) d'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 2, 2, 2)))


<!-- dd:dd-q373 -->

### Problem 373 · independent

Write solve(img) for an (H, W, C) image: unroll the CHANNEL dimension along the height axis — all of channel 0's rows, then channel 1's, then channel 2's — with a trailing singleton axis so the result is ((C·H), W, 1). Pattern: 'h w c -> (c h) w ()'.


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[4]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


<!-- dd:dd-q380 -->

### Problem 380 · independent

Write a function solve(arr) that takes a channels-first batch (b, c, h, w) and returns shape (b*h, w, c): batch merged into height AND the layout converted to channels-last in the same pattern — 'b c h w -> (b h) w c'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (b*h, w, c)."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q392 -->

### Problem 392 · independent

Write a function solve(arr) that takes a channels-first batch (b, c, h, w) and stacks the images VERTICALLY in channels-first layout: return shape (c, b*h, w) via 'b c h w -> c (b h) w'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (c, b*h, w) via 'b c h w -> c (b h) w'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


<!-- dd:dd-q400 -->

### Problem 400 · independent

Write a function solve(arr) that takes a batch (b, c, h, w) and merges batch into width with the batch index INNERMOST: return shape (c, h, w*b) via 'b c h w -> c h (w b)' — the images' columns INTERLEAVE (column 0 of every image, then column 1, ...) instead of sitting side by side. (Contrast '(b w)', which concatenates whole images.)


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (c, h, w*b) via 'b c h w -> c h (w b)' — the images' columns INTERLEAVE (column 0 of every image,"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(2, 1, 2, 2)))


#### Common mistakes

- **"`(h w)` and `(w h)` give the same flattening."** — Same length,
  different order: left-slow/right-fast. The task's phrase "row-major" /
  "column by column" / "X's block first" picks the order for you.
- **"Axes must be adjacent to merge."** — The pattern happily merges
  distant axes ('b h w c -> h (b w) c'); einops inserts the implied
  transpose. By hand you'd have to permute them together first — that
  two-step is exactly what the pattern hides.
- **"Merging loses information."** — It's a pure relabeling; every element
  keeps a unique address. The inverse operation (splitting, next KP)
  recovers the original — provided you remember one of the factor sizes.


<!-- dd:dd-kp-einops-split-axes -->

## Splitting axes with named factors

`einops.split-axes`


Parentheses on the **input** side SPLIT an axis into factors — the exact
inverse of merging:

> `'c (h w) -> c h w', h=h`
> — the length-h·w axis is declared to be h blocks of w, and unpacked.

The new ingredients:

1. **You must tell einops the factor sizes it can't infer.** An axis of
   length 12 could be (h=2, w=6), (3, 4), (4, 3)… — so sizes arrive as
   keyword arguments. Give any factors such that the rest are forced —
   for a two-way split, ONE keyword suffices (`h=3` fixes w = 12/3).
2. **Order inside the parens declares how the axis was PACKED** — left
   slow, right fast, same convention as merging. `(h w)` says "this axis
   is h blocks, each of length w". Splitting with the wrong order doesn't
   error (sizes may still divide) — it unpacks garbage. The task's
   description of how the data was laid out ("row-major tiles", "group
   index slowest/fastest") is the ground truth for the order.
3. **Split and merge combine in one pattern** — the signature einops move.
   `'(b w) ... -> b ...'` unpacks; `'... -> ... (h p)'` repacks;
   `'(h w) p1 p2 c -> (h p1) (w p2) c'` does both at once (that one
   reassembles an image from its tile stack — split the tile index into
   grid coordinates, then merge each with its within-tile axis).

Reading a split-merge pattern: first find every parenthesized group on the
left (what gets unpacked, and in what packing order), then on the right
(what gets packed). The names in the middle just carry through.


Task: restore a flattened image given its height; split a sequence into
chunks; reassemble tiles into an image.


In [ ]:
import torch as t
import einops

# Round-trip: flatten (merge), then restore (split) with one known factor.
img = t.arange(12).reshape(3, 2, 2)                  # (c, h, w)
flat = einops.rearrange(img, 'c h w -> c (h w)')      # (3, 4)
back = einops.rearrange(flat, 'c (h w) -> c h w', h=2)
assert t.equal(back, img)                       # perfect inverse

# Split a sequence into p-token segments: t = n segments of length p.
seq = t.arange(24).reshape(2, 6, 2)                  # (b, t=6, d)
chunks = einops.rearrange(seq, 'b (n p) d -> b n p d', p=3)
assert chunks.shape == (2, 2, 3, 2)
assert t.equal(chunks[0, 0], seq[0, :3])       # first 3 tokens

# Split AND merge at once: tile stack -> image.
# 6 tiles of shape (2, 2), listed row-major from a (2x3)-tile image.
tiles = t.arange(24).reshape(6, 2, 2)                # ((h w), p1, p2)
image = einops.rearrange(tiles, '(h w) p1 p2 -> (h p1) (w p2)', h=2)
assert image.shape == (4, 6)                          # (2*2, 3*2)
# Tile 0 occupies the top-left 2x2 block:
assert image[:2, :2].tolist() == tiles[0].tolist()
print("merge then split round-trips:", bool(t.equal(back, img)))
print("sequence", tuple(seq.shape), "-> chunks", tuple(chunks.shape),
      "| chunk 0 =", chunks[0, 0].tolist())
print("6 tiles -> one", tuple(image.shape), "image:")
print(image)




Why each step:

1. The round-trip (`back == img`) is the defining property of split-as-
   inverse-of-merge, and doubles as your self-test recipe: whenever a split
   pattern feels shaky, merge it back and compare.
2. In the chunking, `p=3` (not n) is given — either works, and choosing the
   one the task names ("segments of length p") keeps the code aligned with
   the prose.
3. The tile reassembly deserves slow reading: `(h w)` unpacks the tile
   index into grid row/column (row-major, hence h slow); then `(h p1)`
   merges grid-row with within-tile-row. Two coordinate systems zipped
   together — one pattern, no loops, no arithmetic on indices.


In [ ]:
import torch as t
import einops

# Round-trip: flatten (merge), then restore (split) with one known factor.
img = t.arange(12).reshape(3, 2, 2)                  # (c, h, w)
flat = einops.rearrange(img, 'c h w -> c (h w)')      # (3, 4)
back = einops.rearrange(flat, 'c (h w) -> c h w', h=2)
assert t.equal(back, img)                       # perfect inverse

# Split a sequence into p-token segments: t = n segments of length p.
seq = t.arange(24).reshape(2, 6, 2)                  # (b, t=6, d)
chunks = einops.rearrange(seq, 'b (n p) d -> b n p d', p=3)
assert chunks.shape == (2, 2, 3, 2)
assert t.equal(chunks[0, 0], seq[0, :3])       # first 3 tokens

# Split AND merge at once: tile stack -> image.
# 6 tiles of shape (2, 2), listed row-major from a (2x3)-tile image.
tiles = t.arange(24).reshape(6, 2, 2)                # ((h w), p1, p2)
image = einops.rearrange(tiles, '(h w) p1 p2 -> (h p1) (w p2)', h=2)
assert image.shape == (4, 6)                          # (2*2, 3*2)
# Tile 0 occupies the top-left 2x2 block:
assert image[:2, :2].tolist() == tiles[0].tolist()
print("merge then split round-trips:", bool(t.equal(back, img)))
print("sequence", tuple(seq.shape), "-> chunks", tuple(chunks.shape),
      "| chunk 0 =", chunks[0, 0].tolist())
print("6 tiles -> one", tuple(image.shape), "image:")
print(image)


<!-- dd:dd-q390 -->

### Problem 390 · faded

Restore (c, h·w) to (c, h, w), given h.


In [ ]:
import torch as t
import einops

def solve(flat, h):
    """Undo the per-channel flatten: declare how the axis was packed."""
    return einops.rearrange(flat, '_____', h=h)


<!-- dd:dd-q315 -->

### Problem 315 · guided

Write a function solve(seq, p) that takes a batch of sequences of shape (b, t, d) — t divisible by p — and splits each sequence into segments of length p: return shape (b, t//p, p, d), the einops pattern 'b (n p) d -> b n p d'.


<details>
<summary>Hints</summary>

1. (b, t, d) with t divisible by p → (b, t/p, p, d): the time axis splits
   into (segments × length-p).
2. Which factor does the task hand you? Pass it as the keyword.
3. `'b (n p) d -> b n p d', p=p` — n is inferred.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(seq, p):
    """Return shape (b, t//p, p, d), the einops pattern 'b (n p) d -> b n p d'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 4, 2), 2))


<!-- dd:dd-q337 -->

### Problem 337 · independent

Write a function solve(merged, b) that takes an image strip of shape (h, b*w, c) — b images concatenated side by side — and SPLITS it back into the batch: return shape (b, h, w, c) via the inverse pattern 'h (b w) c -> b h w c'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(merged, b):
    """Return shape (b, h, w, c) via the inverse pattern 'h (b w) c -> b h w c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 4, 2), 2))


<!-- dd:dd-q320 -->

### Problem 320 · independent

Write solve(imgs) for a (B, H, W, C) batch: divide each image into two equal halves along HEIGHT and stack the halves into the batch axis — output (2·B, H/2, W, C) with all the TOP halves first, then all the bottom halves. Pattern: 'b (two h) w c -> (two b) h w c' with two=2.


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
imgs = arr

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


<!-- dd:dd-q393 -->

### Problem 393 · independent

Write solve(img) for a (C, H, W) image whose C is EVEN: treat the channel axis as (p × two) pairs and split OUT the pair-member axis to the front — output (2, C/2, H, W) where result[0] holds the even-pair members. Pattern: '(p two) h w -> two p h w' with two=2.


In [ ]:
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
img = t.repeat_interleave(arr[5], 2, dim=0)

def solve(img):
    # Write your solution here
    return None

print(solve(img))


<!-- dd:dd-q331 -->

### Problem 331 · independent

Write a function solve(arr, r) that takes a channels-first batch (b, c, h, w), keeps only the EVEN-indexed images arr[0], arr[2], ... (assume that count is divisible by r), and tiles them into an r-row grid of shape (c, r*h, (count//r)*w) — a slice composed with the grid rearrange '(r n) c h w -> c (r h) (n w)'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, r):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(4, 1, 1, 4), 2))


#### Common mistakes

- **"einops can infer both factors of a split."** — It can infer ONE
  (total ÷ known); the rest are yours to supply as keywords. No keyword,
  no split — the error message will list what's missing.
- **"Wrong paren order in a split will error."** — Only if the sizes fail
  to divide. `(h w)` vs `(w h)` with square-ish factors both "work" and one
  is silently scrambled. The packing order comes from how the data was
  BUILT — reread the task's layout description, then round-trip-test.
- **"Tile reassembly needs index arithmetic."** — The split-merge pattern
  `('(h w) p1 p2 -> (h p1) (w p2)')` IS the index arithmetic, stated
  declaratively. If you're computing offsets by hand around einops, the
  pattern can probably absorb the work.


<!-- dd:dd-kp-einops-singleton-and-lists -->

## Singleton axes and lists as an axis

`einops.singleton-and-lists`


Two small pattern features that finish the rearrange grammar:

**`1` — a literal singleton axis.** Writing `1` in a pattern inserts (on
the right) or consumes (on the left) a length-1 axis:

- `'h w -> 1 h w'` — add a leading channel/batch axis (the einops spelling
  of `x[None]`).
- `'b h w c -> b 1 h w c'` — insert one mid-tensor.
- `'1 h w -> h w'` — squeeze a known singleton, with verification: if that
  axis isn't length 1, einops errors instead of silently squeezing the
  wrong thing.

Keeping a REDUCED axis as a singleton (`'h w c -> 1 w c'` in reduce) also
uses this — the reduce KP picks that up.

**A Python list as the first axis.** Handing einops a LIST of same-shape
tensors makes the list index axis 0 — pattern it like any other axis:

- `einops.rearrange([img_a, img_b], 'b h w c -> h (b w) c')` — two images
  side by side, no explicit t.stack first.
- `'b h w c -> b h w c'` on a list is exactly t.stack: the identity
  pattern, with the list→tensor conversion as the entire point.

Together these subsume t.stack / t.unsqueeze / t.squeeze with the
same pattern language you're already using — one notation for the whole
shape-plumbing toolbox.


Task: stack a list of images into a batch; add a singleton channel axis;
combine both in one pattern.


In [ ]:
import torch as t
import einops

imgs = [t.ones((2, 3, 1)) * i for i in range(4)]   # list of (h, w, c)

# List -> batch axis: the identity pattern DOES the stacking.
batch = einops.rearrange(imgs, 'b h w c -> b h w c')
assert batch.shape == (4, 2, 3, 1)
assert batch[2, 0, 0, 0] == 2.0                      # list order preserved

# Singleton insertion: a plain 2-D tensor gains a leading axis.
x2d = t.arange(6).reshape(2, 3)
x3d = einops.rearrange(x2d, 'h w -> 1 h w')
assert x3d.shape == (1, 2, 3)

# Both at once: stack a list AND lay the images out side by side.
pair = [t.zeros((2, 2, 1)), t.ones((2, 2, 1))]
wide = einops.rearrange(pair, 'b h w c -> h (b w) c')
assert wide.shape == (2, 4, 1)
assert wide[0, :, 0].tolist() == [0.0, 0.0, 1.0, 1.0]  # a then b, left to right

# Squeeze with verification: consuming a '1' that isn't there fails loudly.
try:
    einops.rearrange(x2d, '1 h w -> h w')            # x2d is 2-D — no 1 axis
    raised = False
except Exception as err:
    raised = True
    print("squeezing an axis that isn't there ->", type(err).__name__)
assert raised

print("list of 4 images -> batch", tuple(batch.shape),
      "| image 2's first pixel:", batch[2, 0, 0, 0].item())
print("singleton insert:", tuple(x2d.shape), "->", tuple(x3d.shape))
print("stack + side by side:", tuple(wide.shape), "| row 0 =",
      wide[0, :, 0].tolist())




Why each step:

1. The identity pattern on a list looks like a no-op and isn't — the
   conversion is the operation. Reading einops code, remember the input
   TYPE is part of the semantics.
2. The combined example is the idiom to keep: list-stack + merge in one
   declarative step replaces stack-then-rearrange chains. Note b defaults
   to slow in `(b w)`: first list element leftmost.
3. The verified squeeze failing on 2-D data is einops' shape checking
   again — `'1 h w -> h w'` documents an EXPECTATION about the input, and
   the library enforces it. `t.squeeze` would have silently done something.


In [ ]:
import torch as t
import einops

imgs = [t.ones((2, 3, 1)) * i for i in range(4)]   # list of (h, w, c)

# List -> batch axis: the identity pattern DOES the stacking.
batch = einops.rearrange(imgs, 'b h w c -> b h w c')
assert batch.shape == (4, 2, 3, 1)
assert batch[2, 0, 0, 0] == 2.0                      # list order preserved

# Singleton insertion: a plain 2-D tensor gains a leading axis.
x2d = t.arange(6).reshape(2, 3)
x3d = einops.rearrange(x2d, 'h w -> 1 h w')
assert x3d.shape == (1, 2, 3)

# Both at once: stack a list AND lay the images out side by side.
pair = [t.zeros((2, 2, 1)), t.ones((2, 2, 1))]
wide = einops.rearrange(pair, 'b h w c -> h (b w) c')
assert wide.shape == (2, 4, 1)
assert wide[0, :, 0].tolist() == [0.0, 0.0, 1.0, 1.0]  # a then b, left to right

# Squeeze with verification: consuming a '1' that isn't there fails loudly.
try:
    einops.rearrange(x2d, '1 h w -> h w')            # x2d is 2-D — no 1 axis
    raised = False
except Exception as err:
    raised = True
    print("squeezing an axis that isn't there ->", type(err).__name__)
assert raised

print("list of 4 images -> batch", tuple(batch.shape),
      "| image 2's first pixel:", batch[2, 0, 0, 0].item())
print("singleton insert:", tuple(x2d.shape), "->", tuple(x3d.shape))
print("stack + side by side:", tuple(wide.shape), "| row 0 =",
      wide[0, :, 0].tolist())


<!-- dd:dd-q361 -->

### Problem 361 · faded

A Python list of (h, w, c) images → one (b, h, w, c) batch.


In [ ]:
import torch as t
import einops

def solve(imgs):
    """Stack the list: the list index becomes axis b."""
    return einops.rearrange(imgs, '_____')


<!-- dd:dd-q360 -->

### Problem 360 · guided

Write a function solve(x2d) that takes a plain 2-D tensor (h, w) and returns the 3-D version (1, h, w) with a leading singleton channel axis — 'h w -> () h w'.


<details>
<summary>Hints</summary>

1. (h, w) → (1, h, w): nothing moves; one axis appears.
2. The literal `1` on the output side inserts it.
3. `'h w -> 1 h w'`.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x2d):
    """Return the 3-D version (1, h, w) with a leading singleton channel axis — 'h w -> () h w'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 3))))


<!-- dd:dd-q358 -->

### Problem 358 · independent

Write a function solve(arr) that takes a channels-last batch (b, h, w, c) and INSERTS a singleton axis right after the batch axis: return shape (b, 1, h, w, c) via the '()' output group — 'b h w c -> b () h w c'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr):
    """Return shape (b, 1, h, w, c) via the '()' output group — 'b h w c -> b () h w c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.ones((2, 2, 2, 1))))


<!-- dd:dd-q374 -->

### Problem 374 · independent

Write a function solve(imgs) that takes a PYTHON LIST of channels-last images (h, w, c) and concatenates them SIDE BY SIDE: return shape (h, len(list)*w, c) — the list index merging straight into width via 'b h w c -> h (b w) c'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(imgs):
    """Return shape (h, len(list)*w, c) — the list index merging straight into width via 'b h w c -> h (b w) c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve([t.zeros((2, 1, 1)), t.ones((2, 1, 1))]))


<!-- dd:dd-q376 -->

### Problem 376 · independent

Write solve(img_a, img_b) for two identical-shape (H, W, C) images: place them SIDE BY SIDE into one (H, 2·W, C) image, img_a on the left. Pattern: einops.rearrange([img_a, img_b], 'b h w c -> h (b w) c').


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_a = arr[0]
img_b = arr[0]

def solve(img_a, img_b):
    # Write your solution here
    return None

print(solve(img_a, img_b))


<!-- dd:dd-q333 -->

### Problem 333 · independent

Write a function solve(tensors) that takes a PYTHON LIST of channels-first images, each of shape (c, h, w), and returns a single channels-LAST batch tensor of shape (len(list), h, w, c) — einops.rearrange applied directly to the list stacks it as the new leading axis.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(tensors):
    """Return a single channels-LAST batch array of shape (len(list), h, w, c) — einops.rearrange applied directly to"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve([t.ones((1, 2, 2)), t.zeros((1, 2, 2))]))


<!-- dd:dd-q334 -->

### Problem 334 · independent

Write solve(img_a, img_b) for two identical-shape (H, W, C) images: produce a single (2·H, W, C) image whose rows INTERLEAVE the two inputs — row 0 of a, row 0 of b, row 1 of a, row 1 of b, … Pattern: einops.rearrange([img_a, img_b], 'b h w c -> (h b) w c').


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_a = arr[1]
img_b = arr[1]

def solve(img_a, img_b):
    # Write your solution here
    return None

print(solve(img_a, img_b))


<!-- dd:dd-q365 -->

### Problem 365 · independent

Write a function solve(tensors) that takes a PYTHON LIST of channels-first images (c, h, w) and returns a single tensor of shape (h, w, c, len(list)) — stacked AND rearranged so the list axis lands LAST: rearrange(list, 'b c h w -> h w c b').


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(tensors):
    """Return a single array of shape (h, w, c, len(list)) — stacked AND rearranged so the list axis lands LAST."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve([t.ones((1, 2, 2)), t.zeros((1, 2, 2))]))


#### Common mistakes

- **"I must t.stack a list before einops can touch it."** — A list of
  same-shape tensors is accepted directly; its index becomes the first
  axis. The stack is the pattern's job.
- **"`1` in a pattern is a size I'm asserting for a normal axis."** — It's
  a LITERAL singleton: inserted on the right, consumed (with verification)
  on the left. Naming it (like 'c') instead would bind a real axis.
- **"Squeezing with einops is overkill — t.squeeze is fine."** —
  t.squeeze removes ALL singletons, including ones you didn't expect
  (a batch that happens to be size 1!). `'1 h w -> h w'` removes exactly
  the declared one and errors otherwise — overkill is the feature.


<!-- dd:dd-kp-einops-grids-montage -->

## Laying batches out as grids

`einops.grids-montage`


"Tile these b images into a g1-row grid" — the montage — is the flagship
split-then-merge pattern:

> `'(g1 g2) h w c -> (g1 h) (g2 w) c', g1=rows`

Read it in two moves:

1. **Split the batch into grid coordinates.** `(g1 g2)` on the left declares
   the batch axis packs g1 rows of g2 images, ROW-MAJOR (g1 slow: images
   0..g2-1 form grid row 0). One keyword fixes both factors.
2. **Merge each grid coordinate with its image dimension.** `(g1 h)`: grid
   row with within-image height — grid row 0's images occupy output rows
   0..h-1. `(g2 w)`: grid column with width. The output is one big
   ((g1·h) × (g2·w)) image.

Every montage variant is a small edit to this template:

- Channels-first data: same idea around the c axis —
  `'(g1 g2) c h w -> c (g1 h) (g2 w)'`.
- A single row of images ("side by side") is the degenerate g1=1 case —
  which collapses to the merge-KP pattern `'b h w c -> h (b w) c'`.
- Column-major filling would be `(g2 g1)` on the left — the packing-order
  question from the split KP, again decided by the task's words
  ("row-major", "grid position (i, j) holds image i·cols + j").

The montage also runs in REVERSE — carving a grid image back into a batch —
by swapping the pattern's sides: `'(g1 h) (g2 w) c -> (g1 g2) h w c'` with
two keywords, since neither factor of each merged axis is inferable alone.


Task: six images into a 3-row × 2-column grid, row-major — verified by
locating specific images.


In [ ]:
import torch as t
import einops

# Six 2x2 single-channel images; image k is constant k (easy to locate).
imgs = t.stack([t.full((2, 2, 1), float(k)) for k in range(6)])
assert imgs.shape == (6, 2, 2, 1)

grid = einops.rearrange(imgs, '(g1 g2) h w c -> (g1 h) (g2 w) c', g1=3)
assert grid.shape == (6, 4, 1)               # (3*2, 2*2, 1)

# Row-major placement: grid row 0 holds images 0,1; row 1 -> 2,3; row 2 -> 4,5.
assert grid[0, 0, 0] == 0.0                  # top-left block = image 0
assert grid[0, 2, 0] == 1.0                  # top-right block = image 1
assert grid[2, 0, 0] == 2.0                  # second row starts image 2
assert grid[4, 2, 0] == 5.0                  # bottom-right = image 5

# Reverse: carve the montage back into the batch. Each merged input axis
# hides two unknowns, so each group needs one keyword: g1 (fixes h) AND
# g2 (fixes w).
back = einops.rearrange(grid, '(g1 h) (g2 w) c -> (g1 g2) h w c', g1=3, g2=2)
assert t.equal(back, imgs)
print("6 images", tuple(imgs.shape), "-> montage", tuple(grid.shape))
print(grid[:, :, 0])          # each image is a constant block, so read them off
print("carved back to the batch exactly:", bool(t.equal(back, imgs)))




Why each step:

1. Constant-valued test images turn placement checking into value lookups:
   `grid[0, 2]` sitting in grid-row 0, grid-col 1 must equal image 1 under
   row-major packing. Build such fixtures whenever a layout task confuses
   you — arange or constants, never random.
2. The g1=3 keyword does double duty: fixes g2=2 AND documents "3 rows" —
   matching the task's phrasing decides WHICH factor you pass.
3. The reverse pattern needs a keyword PER GROUP (g1 and g2) because each
   merged input axis hides two unknowns and einops solves exactly one
   unknown per parenthesized group.


In [ ]:
import torch as t
import einops

# Six 2x2 single-channel images; image k is constant k (easy to locate).
imgs = t.stack([t.full((2, 2, 1), float(k)) for k in range(6)])
assert imgs.shape == (6, 2, 2, 1)

grid = einops.rearrange(imgs, '(g1 g2) h w c -> (g1 h) (g2 w) c', g1=3)
assert grid.shape == (6, 4, 1)               # (3*2, 2*2, 1)

# Row-major placement: grid row 0 holds images 0,1; row 1 -> 2,3; row 2 -> 4,5.
assert grid[0, 0, 0] == 0.0                  # top-left block = image 0
assert grid[0, 2, 0] == 1.0                  # top-right block = image 1
assert grid[2, 0, 0] == 2.0                  # second row starts image 2
assert grid[4, 2, 0] == 5.0                  # bottom-right = image 5

# Reverse: carve the montage back into the batch. Each merged input axis
# hides two unknowns, so each group needs one keyword: g1 (fixes h) AND
# g2 (fixes w).
back = einops.rearrange(grid, '(g1 h) (g2 w) c -> (g1 g2) h w c', g1=3, g2=2)
assert t.equal(back, imgs)
print("6 images", tuple(imgs.shape), "-> montage", tuple(grid.shape))
print(grid[:, :, 0])          # each image is a constant block, so read them off
print("carved back to the batch exactly:", bool(t.equal(back, imgs)))


<!-- dd:dd-q389 -->

### Problem 389 · faded

Six images → 3×2 grid, row-major.


In [ ]:
import torch as t
import einops

def solve(imgs):
    """(6, h, w, c) -> ((3h), (2w), c), images placed row-major."""
    return einops.rearrange(imgs, '_____', g1=3)


<!-- dd:dd-q364 -->

### Problem 364 · guided

Write a function solve(arr, b1) that takes a channels-LAST batch (b, h, w, c) — b divisible by b1 — and tiles the images into a b1-row montage: return shape (b1*h, (b//b1)*w, c) via '(b1 b2) h w c -> (b1 h) (b2 w) c'. (The companion drill does this for channels-first input.)


<details>
<summary>Hints</summary>

1. A channels-LAST batch (b, h, w, c), b divisible by b1, tiled into a
   b1-row montage — the template with different names.
2. Which factor is given (b1 rows), which inferred?
3. `'(b1 b2) h w c -> (b1 h) (b2 w) c', b1=b1`.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, b1):
    """Return shape (b1*h, (b//b1)*w, c) via '(b1 b2) h w c -> (b1 h) (b2 w) c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(4, 1, 2, 1), 2))


<!-- dd:dd-q329 -->

### Problem 329 · independent

Write a function solve(arr, g1) that takes a channels-first batch of shape (b, c, h, w) — b divisible by g1 — and tiles the images into a g1-row grid: return shape (c, g1*h, (b//g1)*w) via '(g1 g2) c h w -> c (g1 h) (g2 w)', batch order filling each row left to right.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, g1):
    """Return shape (c, g1*h, (b//g1)*w) via '(g1 g2) c h w -> c (g1 h) (g2 w)', batch order filling each row left to"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(4, 1, 1, 2), 2))


<!-- dd:dd-q382 -->

### Problem 382 · independent

Write solve(imgs) for a (6, H, W, C) channels-last batch: split the batch into a 3-row × 2-column grid, row-major, producing one ((3·H), (2·W), C) image. Pattern: '(r nc) h w ch -> (r h) (nc w) ch' with r=3.


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
imgs = arr

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


<!-- dd:dd-q318 -->

### Problem 318 · independent

Write solve(imgs) for a (12, C, H, W) batch of channels-first images: tile them into a single CHW image laid out as a 4-row × 3-column grid (row-major: image i lands at row i//3, column i%3). Pattern: '(g1 g2) c h w -> c (g1 h) (g2 w)' with g1=4.


In [ ]:
import torch as t
import einops

_base = t.tensor(np.load('/delta_numbers.npy'))
imgs = t.cat([_base] * (-(-12 // _base.shape[0])))[:12]

def solve(imgs):
    # Write your solution here
    return None

print(solve(imgs))


<!-- dd:dd-q381 -->

### Problem 381 · independent

Write solve(img_a, img_b, img_c, img_d) for four identical-shape (H, W, C) images: combine them into one (2·H, 2·W, C) image laid out as a 2×2 grid, row-major (a top-left, b top-right, c bottom-left, d bottom-right). Pattern: einops.rearrange([a, b, c, d], '(r nc) h w ch -> (r h) (nc w) ch', r=2).


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_a = arr[0]
img_b = arr[0]
img_c = arr[0]
img_d = arr[0]

def solve(img_a, img_b, img_c, img_d):
    # Write your solution here
    return None

print(solve(img_a, img_b, img_c, img_d))


<!-- dd:dd-q322 -->

### Problem 322 · independent

Write solve(x, hs, ws) for a (B, C, H, W) batch and integer subgrid factors: carve each image's height into hs strips and width into ws strips, moving the (hs, ws) subgrid indices OUT into the batch axis — output (hs·ws·B, C, H/hs, W/ws), subgrid-major then batch. Pattern: 'b c (h hs) (w ws) -> (hs ws b) c h w'.


In [ ]:
import torch as t
import einops

arr = t.tensor(np.load('/delta_numbers.npy'))
x = arr[:2]
hs = 2
ws = 2

def solve(x, hs, ws):
    # Write your solution here
    return None

print(solve(x, hs, ws))


<!-- dd:dd-q371 -->

### Problem 371 · independent

Write a function solve(y, hs, ws) that takes a tensor of shape (hs*ws*b, c, h, w) — spatial subgrids packed into the batch axis, subgrid index slowest — and UNPACKS them back into space: return shape (b, c, h*hs, w*ws) via '(hs ws b) c h w -> b c (h hs) (w ws)'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(y, hs, ws):
    """Return shape (b, c, h*hs, w*ws) via '(hs ws b) c h w -> b c (h hs) (w ws)'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(4, 1, 1, 1), 2, 2))


<!-- dd:dd-q531 -->

### Problem 531 · independent

Write a function solve(imgs, rows) that takes a channels-last batch of images with shape (b, h, w, c) and a grid row count, and returns the single montage image of shape ((rows*h), (cols*w), c) holding all b images — filled COLUMN-major, so image k lands at grid row k % rows and grid column k // rows. b is divisible by rows. This is the row-major montage pattern with one edit, and the edit is on the INPUT side: which of the two batch factors counts slowly decides the filling order, so swap the split order rather than renaming anything.


In [ ]:
import torch as t
import einops

def solve(imgs, rows):
    """(b, h, w, c) -> ((rows*h), (cols*w), c), images filled COLUMN-major."""
    return None


# Example run — the grader calls solve() with several different inputs,
# including edge cases. Your function must work for all of them.
example = (t.stack([t.full((2, 2, 1), float(k)) for k in range(6)]), 3)
print(solve(*example))


#### Common mistakes

- **"Montages need a loop pasting images into a canvas."** — The
  split-merge pattern is the whole operation. If you're computing paste
  offsets, the pattern replaces that code.
- **"g1 is rows because it's named g1."** — It's rows because it's the SLOW
  factor of the batch split AND merges with h. Rename freely; position does
  the work. (Corollary: to fill column-major, swap the split order, not the
  names.)
- **"The grid pattern only works for images."** — Any batch × per-item-2D
  data montages the same way; and the same split-merge shape reappears in
  patches, space-to-depth, and pooling. Learn it as geometry, not as an
  image trick.


<!-- dd:dd-kp-einops-patches-space-depth -->

## Patches, space-to-depth, depth-to-space

`einops.patches-space-depth`


Vision architectures constantly trade SPACE for DEPTH (or batch): cut each
image into little patches and treat them as tokens (ViT), fold pixel blocks
into channels (space-to-depth), or unfold them back (depth-to-space /
pixel-shuffle). All are the grid pattern of the last KP with the roles
recast:

**Patch extraction** — split each spatial axis into (blocks × within-block),
then pull the block coordinates out as a patch index:

> `'(h p1) (w p2) c -> (h w) p1 p2 c', p1=P, p2=P`

Height splits into h blocks of p1 rows; width likewise; merging (h w)
row-major gives the patch list. **Reassembly is the same pattern reversed**
— it was your grid-KP faded exercise (q323), now recognized as
depth-to-space's cousin.

**Space-to-depth** — the block coordinates fold into the CHANNEL axis
instead of a patch index:

> `'b c (h p) (w q) -> b (c p q) h w', p=P, q=P`

Each p×q pixel block's values become extra channels; spatial dims shrink by
P, channels grow by P². (The output channel packing order — c slow, then p,
then q — is dictated by the paren order; tasks specify theirs.)

**Depth-to-space** mirrors it: `'b (c p q) h w -> b c (h p) (w q)'` — the
channel axis DECLARES its factorization, and blocks unfold back into space.
This is pixel-shuffle upsampling.

The discipline for all of them: write the INPUT side to describe how the
data is actually packed (which factor is slow), the OUTPUT side to describe
what the task wants — then hand einops the block sizes. When the packing
order is ambiguous in your head, build a tiny arange example and round-trip.


Task: extract 2×2 patches from an image, reassemble them, and run a
space-to-depth.


In [ ]:
import torch as t
import einops

img = t.arange(16.0).reshape(4, 4, 1)     # (H, W, c=1), values = positions

# PATCHES: split H into 2 blocks of 2, W likewise; block coords -> patch axis.
patches = einops.rearrange(img, '(h p1) (w p2) c -> (h w) p1 p2 c', p1=2, p2=2)
assert patches.shape == (4, 2, 2, 1)
# Patch 0 = top-left 2x2 block, row-major patch order:
assert patches[0, :, :, 0].tolist() == [[0.0, 1.0], [4.0, 5.0]]
assert patches[1, 0, 0, 0] == 2.0          # patch 1 starts at column 2

# REASSEMBLE: the same pattern, sides swapped (this is q323's shape).
back = einops.rearrange(patches, '(h w) p1 p2 c -> (h p1) (w p2) c', h=2)
assert t.equal(back, img)

# SPACE-TO-DEPTH on a batch: blocks fold into channels; H, W halve.
x = t.arange(32.0).reshape(1, 2, 4, 4)    # (b, c=2, H, W)
s2d = einops.rearrange(x, 'b c (h p) (w q) -> b (c p q) h w', p=2, q=2)
assert s2d.shape == (1, 8, 2, 2)           # channels x4, spatial /2
# The new channel block for output pixel (0,0) holds input block [0:2, 0:2]:
assert s2d[0, :4, 0, 0].tolist() == [0.0, 1.0, 4.0, 5.0]
print("image", tuple(img.shape), "-> patches", tuple(patches.shape))
print("patch 0 =", patches[0, :, :, 0].tolist(),
      "| patch 1 starts at", patches[1, 0, 0, 0].item())
print("reassembled to the original:", bool(t.equal(back, img)))
print("space-to-depth", tuple(x.shape), "->", tuple(s2d.shape),
      "(channels x4, spatial /2)")
print("pixel (0,0)'s new channels:", s2d[0, :4, 0, 0])




Why each step:

1. Position-valued pixels make each check readable: patch 1 starting at
   value 2.0 confirms row-major patch order and a correct width split.
   This fixture technique is how to debug ANY packing dispute with einops.
2. The reassembly line being the extraction line reversed (with the
   keyword moving to the other side's unknowns) cements the symmetry —
   patches/grids/s2d are one bijection family, direction chosen by which
   side carries the parens you're UNPACKING.
3. In space-to-depth, verify the channel packing: c slow, p, then q — the
   four values 0,1,4,5 are block (0,0) in row-major order, sitting after
   channel 0's... here c=... the first 4 output channels come from input
   channel 0. Reading packed channel layouts element-by-element once
   inoculates against the classic s2d ordering bug.


In [ ]:
import torch as t
import einops

img = t.arange(16.0).reshape(4, 4, 1)     # (H, W, c=1), values = positions

# PATCHES: split H into 2 blocks of 2, W likewise; block coords -> patch axis.
patches = einops.rearrange(img, '(h p1) (w p2) c -> (h w) p1 p2 c', p1=2, p2=2)
assert patches.shape == (4, 2, 2, 1)
# Patch 0 = top-left 2x2 block, row-major patch order:
assert patches[0, :, :, 0].tolist() == [[0.0, 1.0], [4.0, 5.0]]
assert patches[1, 0, 0, 0] == 2.0          # patch 1 starts at column 2

# REASSEMBLE: the same pattern, sides swapped (this is q323's shape).
back = einops.rearrange(patches, '(h w) p1 p2 c -> (h p1) (w p2) c', h=2)
assert t.equal(back, img)

# SPACE-TO-DEPTH on a batch: blocks fold into channels; H, W halve.
x = t.arange(32.0).reshape(1, 2, 4, 4)    # (b, c=2, H, W)
s2d = einops.rearrange(x, 'b c (h p) (w q) -> b (c p q) h w', p=2, q=2)
assert s2d.shape == (1, 8, 2, 2)           # channels x4, spatial /2
# The new channel block for output pixel (0,0) holds input block [0:2, 0:2]:
assert s2d[0, :4, 0, 0].tolist() == [0.0, 1.0, 4.0, 5.0]
print("image", tuple(img.shape), "-> patches", tuple(patches.shape))
print("patch 0 =", patches[0, :, :, 0].tolist(),
      "| patch 1 starts at", patches[1, 0, 0, 0].item())
print("reassembled to the original:", bool(t.equal(back, img)))
print("space-to-depth", tuple(x.shape), "->", tuple(s2d.shape),
      "(channels x4, spatial /2)")
print("pixel (0,0)'s new channels:", s2d[0, :4, 0, 0])


<!-- dd:dd-q323 -->

### Problem 323 · faded

Reassemble a row-major tile stack into the image.


In [ ]:
import torch as t
import einops

def solve(patches, h, w):
    """(h*w, p1, p2, c) tiles, row-major -> ((h p1), (w p2), c) image."""
    return einops.rearrange(patches, '_____', h=h, w=w)


<!-- dd:dd-q313 -->

### Problem 313 · guided

Write a function solve(x, p) that takes a 4-D channels-first batch x of shape (b, c, h, w) — h and w divisible by p — and performs SPACE-TO-DEPTH with block size p: pack each non-overlapping p x p spatial block into the channel dimension. The result has shape (b, c*p*p, h//p, w//p), matching einops.rearrange(x, 'b c (h p1) (w p2) -> b (c p1 p2) h w', p1=p, p2=p).


<details>
<summary>Hints</summary>

1. Space-to-depth on (b, c, h, w) with block size p: spatial dims shrink by
   p, channels multiply by p².
2. Split each spatial axis into (blocks × p); fold the two p-factors into
   the channel group — the task states the required channel packing order.
3. `'b c (h p1) (w p2) -> b (c p1 p2) h w', p1=p, p2=p` — or the order the
   prompt demands; verify one block.

</details>


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, p):
    """Implement the described contraction and return the result."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 1, 4, 4), 2))


<!-- dd:dd-q401 -->

### Problem 401 · independent

Write solve(img_hwc) for an (H, W, C) image with H and W divisible by 3: extract all non-overlapping 3×3 patches into a ((H/3 · W/3), 3, 3, C) tensor, patches ordered row-major. Pattern: '(h p1) (w p2) c -> (h w) p1 p2 c' with p1=3, p2=3.


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img_hwc = arr[0]

def solve(img_hwc):
    # Write your solution here
    return None

print(solve(img_hwc))


<!-- dd:dd-q404 -->

### Problem 404 · independent

Write a function solve(img, p) that takes a channels-first image (c, h*p, w*p) and EXTRACTS its non-overlapping p x p patches: return shape (h*w, c, p, p) — patches listed row-major — via 'c (h p1) (w p2) -> (h w) c p1 p2'. (The inverse of patch reassembly.)


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(img, p):
    """Return shape (h*w, c, p, p) — patches listed row-major — via 'c (h p1) (w p2) -> (h w) c p1 p2'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 4, 4), 2))


<!-- dd:dd-q343 -->

### Problem 343 · independent

Write a function solve(arr, p1, p2) that takes a tensor of shape (b, c*p1*p2, h, w) — channel groups ordered with c slowest — and performs DEPTH-TO-SPACE: return shape (b, c, h*p1, w*p2) via 'b (c p1 p2) h w -> b c (h p1) (w p2)', spreading each group of p1*p2 channels over a finer spatial grid.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, p1, p2):
    """Return shape (b, c, h*p1, w*p2) via 'b (c p1 p2) h w -> b c (h p1) (w p2)', spreading each group of p1*p2 chan"""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(4.0).reshape(1, 4, 1, 1), 2, 2))


<!-- dd:dd-q350 -->

### Problem 350 · independent

Write solve(img) for an (H, W, C) image with H and W even: within every non-overlapping 2×2 patch, TRANSPOSE the patch (swap its within-patch row and column). Pattern: '(h p1) (w p2) c -> (h p2) (w p1) c' with p1=2, p2=2. Output shape is unchanged; content moves.


In [ ]:
import torch as t
import einops

arr = t.movedim(t.tensor(np.load('/delta_numbers.npy')), 1, -1)
img = arr[0]

def solve(img):
    # Write your solution here
    return None

print(solve(img))


<!-- dd:dd-q321 -->

### Problem 321 · independent

Write a function solve(x, h1, w1) that takes a tensor of shape (b, h1*w1*c, h, w) and redistributes channel groups into space: return shape (b, c, h*h1, w*w1) via 'b (h1 w1 c) h w -> b c (h h1) (w w1)' — each of the h1*w1 channel groups becomes one sub-position of an enlarged pixel grid.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(x, h1, w1):
    """Return shape (b, c, h*h1, w*w1) via 'b (h1 w1 c) h w -> b c (h h1) (w w1)' — each of the h1*w1 channel groups """
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(8.0).reshape(1, 8, 1, 1), 4, 2))


<!-- dd:dd-q395 -->

### Problem 395 · independent

Write solve(x, h1, w1) for a (B, C, H, W) batch: space-to-depth — carve each spatial map into h1×w1 blocks and fold the block-position indices INTO the channel axis, output (B, h1·w1·C, H/h1, W/w1) with channel order (h1, w1, c). Pattern: 'b c (h h1) (w w1) -> b (h1 w1 c) h w'.


In [ ]:
import torch as t
import einops

x = t.arange(2 * 12 * 8 * 8).reshape(2, 12, 8, 8)
h1 = 4
w1 = 2

def solve(x, h1, w1):
    # Write your solution here
    return None

print(solve(x, h1, w1))


<!-- dd:dd-q398 -->

### Problem 398 · independent

Write a function solve(patches, h, w) that takes a patch stack of shape (h*w, c, p1, p2) — channels-FIRST patches listed row-major — and reassembles the image in channels-first form: return shape (c, h*p1, w*p2) via '(h w) c p1 p2 -> c (h p1) (w p2)'. (The companion drill reassembles channels-last patches.)


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(patches, h, w):
    """Return shape (c, h*p1, w*p2) via '(h w) c p1 p2 -> c (h p1) (w p2)'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(4, 1, 2, 2), 2, 2))


<!-- dd:dd-q403 -->

### Problem 403 · independent

Write a function solve(arr, w4) that takes a BHWC batch (b, h, w*w4, c) and folds width chunks into height: factor the width axis as (w, w4) with w4 INNER, then merge w4 into height AHEAD of h — return shape (b, w4*h, w, c) via 'b h (w w4) c -> b (w4 h) w c'.


In [ ]:
import torch as t
import einops
from einops import rearrange, reduce, repeat

def solve(arr, w4):
    """Return shape (b, w4*h, w, c) via 'b h (w w4) c -> b (w4 h) w c'."""
    return None


# Example run — the grader calls solve() with several inputs.
print(solve(t.arange(16.0).reshape(1, 1, 8, 2), 4))


#### Common mistakes

- **"Patch extraction needs sliding-window machinery."** — NON-overlapping
  patches are a pure reshape (split + merge); no windows, no copies of
  copies. Sliding (overlapping) windows are `x.unfold(...)`'s job — different
  task, check the word "non-overlapping".
- **"Space-to-depth loses spatial information."** — It's a bijection: every
  pixel gets a unique (channel, position) address, and depth-to-space
  inverts it exactly. What changes is which axis "sees" the detail.
- **"The channel packing order after s2d doesn't matter."** — Downstream
  code (or the grader) reads channels by index; (c p q) vs (p q c) are
  different tensors with equal shapes. The paren order IS the file format —
  get it from the task, verify with a position-valued example.
